In [1]:
import polars as pl

df = pl.read_csv(
    "megascale_dataset_with_ddg.csv",
    infer_schema_length=10000
)
df

original_seq_full,mutated_seq_full,deltaG,deltaG_wt,ddG,mut_type,reverse
str,str,f64,f64,f64,str,bool
"""SAGGTYTWNTKEEAKQAFKELLKEKRVPSN…","""SAGGTYTWNTKEEAKQAFKELLKEKPVPSN…",0.609712,2.23667,-1.626958,"""R22P""",false
"""SAGGSAGGSAGGHEITLHINGRRVKLRFRD…","""SAGGSAGGSAGGHEITLHINGRRVKLRFTD…",3.58818,4.440977,-0.852797,"""R17T""",false
"""SAGGSAGGSKDPKFEAAYDFPGSGSSSELP…","""SAGGSAGGSKDPKFEAAMDFPGSGSSSELP…",1.333048,2.501842,-1.168794,"""Y9M:K24Q""",false
"""SAGGNKASVVANQLIPINTALTLIMMKAEV…","""SAGGNKASVVANQLIPINTALTLIMMKAEV…",3.863437,4.161053,-0.297616,"""K39N""",false
"""SAGGSAGWVPTKREEKYGVAFYNYDARGAD…","""SAGGSAGWVPTKREEKYGVAFYNYDARGAD…",0.75604,2.212657,-1.456617,"""D31M:T47G""",false
…,…,…,…,…,…,…
"""SAGGNQASVVANQLIPINTALTLVMMRSEE…","""SAGGNQASVVANQLIPINTALTLVMMRSEV…",7.052293,6.031509,-1.020783,"""V26E""",true
"""SAGGSAGGSAQGDIVVALYPEDGIHPDDLS…","""SAGGSAGGSAQGDIVVALYPYDGIHPDDLS…",-1.42882,3.932004,5.360824,"""Y11E:Y54I""",true
"""SAGGSAGGMTYKLILNGKTLKGETTTEAVD…","""SAGGSAGGMTYKLILNGKTLKGETTTEAVD…",1.998289,2.698998,0.700709,"""D47K""",true


In [2]:
df["ddG"].describe()

statistic,value
str,f64
"""count""",833828.0
"""null_count""",0.0
"""mean""",-2.6042e-17
"""std""",2.46046
"""min""",-17.674774
"""25%""",-0.962962
"""50%""",0.000002
"""75%""",0.962962
"""max""",17.674774


In [3]:
k_neg = 0.261
k_pos = 0.261
A_neg = 1.0
A_pos = 1.0

sigmoid_expr = (
    pl.when(pl.col("ddG") >= 0)
    .then(A_pos * (2 / (1 + (-k_pos * pl.col("ddG")).exp()) - 1))
    .otherwise(-A_neg * (2 / (1 + (-k_neg * (-pl.col("ddG"))).exp()) - 1))
)

df = df.with_columns(
    sigmoid_expr.alias("normalized_ddG_sigmoid")
)

df.write_csv("megascale_dataset_with_ddg_normalized.csv")
df

original_seq_full,mutated_seq_full,deltaG,deltaG_wt,ddG,mut_type,reverse,normalized_ddG_sigmoid
str,str,f64,f64,f64,str,bool,f64
"""SAGGTYTWNTKEEAKQAFKELLKEKRVPSN…","""SAGGTYTWNTKEEAKQAFKELLKEKPVPSN…",0.609712,2.23667,-1.626958,"""R22P""",false,-0.209184
"""SAGGSAGGSAGGHEITLHINGRRVKLRFRD…","""SAGGSAGGSAGGHEITLHINGRRVKLRFTD…",3.58818,4.440977,-0.852797,"""R17T""",false,-0.110833
"""SAGGSAGGSKDPKFEAAYDFPGSGSSSELP…","""SAGGSAGGSKDPKFEAAMDFPGSGSSSELP…",1.333048,2.501842,-1.168794,"""Y9M:K24Q""",false,-0.151356
"""SAGGNKASVVANQLIPINTALTLIMMKAEV…","""SAGGNKASVVANQLIPINTALTLIMMKAEV…",3.863437,4.161053,-0.297616,"""K39N""",false,-0.038819
"""SAGGSAGWVPTKREEKYGVAFYNYDARGAD…","""SAGGSAGWVPTKREEKYGVAFYNYDARGAD…",0.75604,2.212657,-1.456617,"""D31M:T47G""",false,-0.187832
…,…,…,…,…,…,…,…
"""SAGGNQASVVANQLIPINTALTLVMMRSEE…","""SAGGNQASVVANQLIPINTALTLVMMRSEV…",7.052293,6.031509,-1.020783,"""V26E""",true,-0.13243
"""SAGGSAGGSAQGDIVVALYPEDGIHPDDLS…","""SAGGSAGGSAQGDIVVALYPYDGIHPDDLS…",-1.42882,3.932004,5.360824,"""Y11E:Y54I""",true,0.604106
"""SAGGSAGGMTYKLILNGKTLKGETTTEAVD…","""SAGGSAGGMTYKLILNGKTLKGETTTEAVD…",1.998289,2.698998,0.700709,"""D47K""",true,0.091188


In [4]:
df["normalized_ddG_sigmoid"].describe()

statistic,value
str,f64
"""count""",833828.0
"""null_count""",0.0
"""mean""",8.0102e-18
"""std""",0.274199
"""min""",-0.980353
"""25%""",-0.125009
"""50%""",2.1026e-7
"""75%""",0.125009
"""max""",0.980353


In [7]:
df.filter(pl.col('original_seq_full') == pl.col('mutated_seq_full'))

original_seq_full,mutated_seq_full,deltaG,deltaG_wt,ddG,mut_type,reverse,normalized_ddG_sigmoid
str,str,f64,f64,f64,str,bool,f64


In [13]:
len(unique_seqs_lehner)

430